In [1]:
# ============================================================
# Cell 1 — All Imports
# ============================================================
import pandas as pd
import numpy as np
import requests
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Your local API endpoint (must be running before Cell 4)
API_URL = "http://localhost:8000/analyze"

# The 20 Golden Features your XGBoost model uses
GOLDEN_FEATURES = [
    'Packet Length Variance',
    'Bwd Packet Length Max',
    'Max Packet Length',
    'Total Length of Fwd Packets',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Fwd IAT Std',
    'Fwd Packet Length Max',
    'Bwd Header Length',
    'Fwd Header Length',
    'PSH Flag Count',
    'Flow IAT Std',
    'Init_Win_bytes_backward',
    'Flow IAT Mean',
    'Bwd Packet Length Min',
    'Flow IAT Max',
    'min_seg_size_forward',
    'Min Packet Length',
    'Init_Win_bytes_forward',
    'act_data_pkt_fwd'
]

print("✅ Imports complete")
print(f"   API target: {API_URL}")
print(f"   Golden features: {len(GOLDEN_FEATURES)}")

✅ Imports complete
   API target: http://localhost:8000/analyze
   Golden features: 20


In [2]:
# ============================================================
# Cell 2 — Load Live Traffic CSV
# UPDATE THIS PATH to wherever your file is saved
# ============================================================
import os

# Try common locations automatically
possible_paths = [
    "/Users/kruthikhavishali/Downloads/edge-defense-api/live_traffic/all_live_traffic.csv",
    "/Users/kruthikhavishali/Downloads/all_live_traffic.csv",
    "/Users/kruthikhavishali/Desktop/all_live_traffic.csv",
    "all_live_traffic.csv",  # same folder as notebook
]

CSV_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        CSV_PATH = p
        break

if CSV_PATH is None:
    # Manually set it here if none found
    CSV_PATH = "/Users/kruthikhavishali/Downloads/all_live_traffic.csv"
    print(f"⚠️  Could not auto-find file — using: {CSV_PATH}")
    print("   If wrong, edit CSV_PATH above manually")
else:
    print(f"✅ Found CSV at: {CSV_PATH}")

# Load
df = pd.read_csv(CSV_PATH, encoding='utf-8', encoding_errors='replace', low_memory=False)
df.columns = df.columns.str.strip()

print(f"\n📊 Dataset info:")
print(f"   Total flows : {len(df)}")
print(f"   Total cols  : {len(df.columns)}")
print(f"   Label dist  : {df['label'].value_counts().to_dict()}")

✅ Found CSV at: /Users/kruthikhavishali/Downloads/edge-defense-api/live_traffic/all_live_traffic.csv

📊 Dataset info:
   Total flows : 413
   Total cols  : 347
   Label dist  : {'Unknown': 413}


In [3]:
# ============================================================
# Cell 3 — Feature Mapping (snake_case → CIC-IDS2017 format)
# Your CSV uses different column names than the training data
# This mapping was verified against all 347 columns in your CSV
# ============================================================

COLUMN_MAP = {
    'payload_bytes_variance':    'Packet Length Variance',
    'bwd_payload_bytes_max':     'Bwd Packet Length Max',
    'payload_bytes_max':         'Max Packet Length',
    'fwd_total_payload_bytes':   'Total Length of Fwd Packets',
    'payload_bytes_mean':        'Packet Length Mean',
    'fwd_payload_bytes_mean':    'Fwd Packet Length Mean',
    'fwd_packets_IAT_std':       'Fwd IAT Std',
    'fwd_payload_bytes_max':     'Fwd Packet Length Max',
    'bwd_total_header_bytes':    'Bwd Header Length',
    'fwd_total_header_bytes':    'Fwd Header Length',
    'psh_flag_counts':           'PSH Flag Count',
    'packet_IAT_std':            'Flow IAT Std',
    'bwd_init_win_bytes':        'Init_Win_bytes_backward',
    'packets_IAT_mean':          'Flow IAT Mean',
    'bwd_payload_bytes_min':     'Bwd Packet Length Min',
    'packet_IAT_max':            'Flow IAT Max',
    'fwd_segment_size_min':      'min_seg_size_forward',
    'payload_bytes_min':         'Min Packet Length',
    'fwd_init_win_bytes':        'Init_Win_bytes_forward',
    'fwd_packets_count':         'act_data_pkt_fwd',
}

# Apply renaming
df = df.rename(columns=COLUMN_MAP)

# Verify all 20 are now present
missing = [f for f in GOLDEN_FEATURES if f not in df.columns]
present = [f for f in GOLDEN_FEATURES if f in df.columns]

if missing:
    print(f"❌ Still missing after mapping: {missing}")
    print("   These will be filled with 0 (safe for tree models)")
    for f in missing:
        df[f] = 0.0
else:
    print(f"✅ All 20 Golden Features mapped successfully")

# Extract only the 20 features
df_golden = df[GOLDEN_FEATURES].copy()

# Clean: replace Inf and NaN with 0 (safe for XGBoost)
df_golden = df_golden.replace([float('inf'), float('-inf')], float('nan')).fillna(0)

# Convert all to float to avoid numpy type issues with JSON
df_golden = df_golden.astype(float)

# Verify non-zero data
non_zero = sum(1 for f in GOLDEN_FEATURES if df_golden[f].sum() != 0)
print(f"\n📊 Feature quality check:")
print(f"   Features with real data (non-zero) : {non_zero}/20")
print(f"   Total flows ready to analyze       : {len(df_golden)}")
print(f"\n🔍 Sample flow (row 0):")
for f in GOLDEN_FEATURES:
    val = df_golden.iloc[0][f]
    print(f"   {f:35s}: {val}")

✅ All 20 Golden Features mapped successfully

📊 Feature quality check:
   Features with real data (non-zero) : 13/20
   Total flows ready to analyze       : 413

🔍 Sample flow (row 0):
   Packet Length Variance             : 0.0
   Bwd Packet Length Max              : 0.0
   Max Packet Length                  : 39.0
   Total Length of Fwd Packets        : 63.0
   Packet Length Mean                 : 0.0
   Fwd Packet Length Mean             : 0.0
   Fwd IAT Std                        : 0.0
   Fwd Packet Length Max              : 39.0
   Bwd Header Length                  : 96.0
   Fwd Header Length                  : 96.0
   PSH Flag Count                     : 2.0
   Flow IAT Std                       : 0.0
   Init_Win_bytes_backward            : 2048.0
   Flow IAT Mean                      : 0.0
   Bwd Packet Length Min              : 0.0
   Flow IAT Max                       : 0.0001068115234375
   min_seg_size_forward               : 32.0
   Min Packet Length                  : 0.0

In [4]:
# ============================================================
# Cell 4 — Test API Connection (single flow)
# Make sure uvicorn is running before this cell:
#   cd ~/Downloads/edge-defense-api
#   source venv/bin/activate
#   uvicorn app.main:app --reload
# ============================================================

def analyze_flow(flow_dict):
    """Send one flow to FastAPI and return prediction."""
    # Convert all values to Python native float (not numpy)
    clean_dict = {k: float(v) for k, v in flow_dict.items()}
    payload = {"features": clean_dict}
    try:
        response = requests.post(API_URL, json=payload, timeout=30)
        if response.status_code == 200:
            return response.json()
        else:
            return {
                "error": f"HTTP {response.status_code}",
                "detail": response.text[:300]
            }
    except requests.exceptions.ConnectionError:
        return {"error": "CONNECTION_REFUSED — start backend: uvicorn app.main:app --reload"}
    except requests.exceptions.Timeout:
        return {"error": "TIMEOUT — API took >30s"}
    except Exception as e:
        return {"error": str(e)}

# Test with flow #0
print("🔌 Testing API connection with Flow #0...")
test_flow = df_golden.iloc[0].to_dict()
result = analyze_flow(test_flow)

if "error" in result:
    print(f"\n❌ ERROR: {result['error']}")
    if "detail" in result:
        print(f"   Detail: {result['detail']}")
    print("\n   Fix: Make sure backend is running in terminal:")
    print("   cd ~/Downloads/edge-defense-api")
    print("   source venv/bin/activate")
    print("   uvicorn app.main:app --reload")
else:
    print(f"\n✅ API connected successfully!")
    print(f"   Label      : {result['label']}")
    print(f"   Probability: {result['probability']*100:.3f}%")
    print(f"   Inference  : {result['inference_ms']}ms")
    print(f"   SHAP values: {len(result['shap_values'])} features")
    print(f"   Top feature: {result['feature_names'][result['shap_values'].index(max(result['shap_values']))]}")

🔌 Testing API connection with Flow #0...

✅ API connected successfully!
   Label      : BENIGN
   Probability: 0.035%
   Inference  : 2.3084ms
   SHAP values: 20 features
   Top feature: act_data_pkt_fwd


In [5]:
# ============================================================
# Cell 5 — Analyze All 413 Captured Flows
# Only run after Cell 4 shows ✅ API connected
# ============================================================

print(f"🚀 Analyzing {len(df_golden)} real live network flows...\n")

results      = []
attack_count = 0
benign_count = 0
error_count  = 0
error_sample = None  # save one error for debugging

for i, (idx, row) in enumerate(df_golden.iterrows()):
    flow_dict = row.to_dict()
    result    = analyze_flow(flow_dict)

    if "error" in result:
        error_count += 1
        if error_sample is None:
            error_sample = result  # save first error
        continue

    top_feat = result['feature_names'][
        result['shap_values'].index(max(result['shap_values']))
    ]

    results.append({
        'flow_index':      i,
        'label':           result['label'],
        'prediction':      result['prediction'],
        'probability':     round(result['probability'], 6),
        'inference_ms':    round(result['inference_ms'], 3),
        'top_shap_feature': top_feat.strip()
    })

    if result['prediction'] == 1:
        attack_count += 1
        print(f"  Flow #{i:3d} → ⚠️  ATTACK  ({result['probability']*100:.2f}%) | Top: {top_feat.strip()}")
    else:
        benign_count += 1
        if i % 20 == 0:
            print(f"  Flow #{i:3d} → ✓  BENIGN  ({result['probability']*100:.3f}%)")

    time.sleep(0.05)  # 50ms delay — enough to not overwhelm API

print(f"\n{'='*50}")
print(f"ANALYSIS COMPLETE")
print(f"{'='*50}")
print(f"Total flows analyzed : {len(results)}")
print(f"ATTACK flows         : {attack_count}")
print(f"BENIGN flows         : {benign_count}")
print(f"Errors               : {error_count}")

# Safe division — no ZeroDivisionError
if len(results) > 0:
    print(f"Attack rate          : {attack_count/len(results)*100:.2f}%")
    print(f"Avg inference time   : {sum(r['inference_ms'] for r in results)/len(results):.3f}ms")
else:
    print(f"\n❌ All flows errored — no results to show")
    if error_sample:
        print(f"   First error was: {error_sample}")
    print(f"\n   Debug: re-run Cell 4 and fix the API connection first")

🚀 Analyzing 413 real live network flows...

  Flow #  0 → ✓  BENIGN  (0.035%)
  Flow # 20 → ✓  BENIGN  (0.280%)
  Flow # 40 → ✓  BENIGN  (0.014%)
  Flow # 60 → ✓  BENIGN  (0.584%)
  Flow # 80 → ✓  BENIGN  (0.067%)
  Flow #100 → ✓  BENIGN  (0.001%)
  Flow #120 → ✓  BENIGN  (0.001%)
  Flow #140 → ✓  BENIGN  (0.001%)
  Flow #160 → ✓  BENIGN  (0.372%)
  Flow #180 → ✓  BENIGN  (0.001%)
  Flow #200 → ✓  BENIGN  (0.001%)
  Flow #220 → ✓  BENIGN  (0.003%)
  Flow #240 → ✓  BENIGN  (0.064%)
  Flow #260 → ✓  BENIGN  (0.865%)
  Flow #280 → ✓  BENIGN  (0.006%)
  Flow #300 → ✓  BENIGN  (0.004%)
  Flow #320 → ✓  BENIGN  (0.064%)
  Flow #340 → ✓  BENIGN  (0.001%)
  Flow #360 → ✓  BENIGN  (0.001%)
  Flow #380 → ✓  BENIGN  (0.001%)
  Flow #400 → ✓  BENIGN  (0.001%)

ANALYSIS COMPLETE
Total flows analyzed : 413
ATTACK flows         : 0
BENIGN flows         : 413
Errors               : 0
Attack rate          : 0.00%
Avg inference time   : 1.121ms


In [6]:
# ============================================================
# Cell 6 — Save Results + Final Report
# Only run after Cell 5 completes with results > 0
# ============================================================

if len(results) == 0:
    print("❌ No results to save — fix errors in Cell 5 first")
else:
    results_df = pd.DataFrame(results)

    # Save to live_traffic folder
    import os
    save_dir = os.path.dirname(CSV_PATH)
    save_path = os.path.join(save_dir, 'live_analysis_results.csv')
    results_df.to_csv(save_path, index=False)

    print(f"✅ Results saved to: {save_path}")
    print(f"\n{'='*55}")
    print(f"  LIVE NETWORK ANALYSIS REPORT")
    print(f"{'='*55}")
    print(f"  Total flows     : {len(results_df)}")
    print(f"  ATTACK detected : {attack_count} ({attack_count/len(results_df)*100:.1f}%)")
    print(f"  BENIGN flows    : {benign_count} ({benign_count/len(results_df)*100:.1f}%)")
    print(f"  Avg inference   : {results_df['inference_ms'].mean():.3f}ms")
    print(f"  Max inference   : {results_df['inference_ms'].max():.3f}ms")
    print(f"{'='*55}")

    if attack_count > 0:
        print(f"\n⚠️  ATTACKS DETECTED:")
        attacks = results_df[results_df['prediction'] == 1]
        for _, row in attacks.iterrows():
            print(f"   Flow #{int(row['flow_index']):3d} — "
                  f"{row['probability']*100:.2f}% confidence | "
                  f"Top feature: {row['top_shap_feature']}")
    else:
        print(f"\n✅ All flows BENIGN — network traffic looks clean")
        print(f"   (Normal home/office WiFi traffic is expected to be benign)")

    print(f"\n📄 Columns in saved CSV:")
    print(f"   {list(results_df.columns)}")
    print(f"\n   Upload live_analysis_results.csv to the")
    print(f"   'Live Traffic' tab in your dashboard")
    

✅ Results saved to: /Users/kruthikhavishali/Downloads/edge-defense-api/live_traffic/live_analysis_results.csv

  LIVE NETWORK ANALYSIS REPORT
  Total flows     : 413
  ATTACK detected : 0 (0.0%)
  BENIGN flows    : 413 (100.0%)
  Avg inference   : 1.121ms
  Max inference   : 3.327ms

✅ All flows BENIGN — network traffic looks clean
   (Normal home/office WiFi traffic is expected to be benign)

📄 Columns in saved CSV:
   ['flow_index', 'label', 'prediction', 'probability', 'inference_ms', 'top_shap_feature']

   Upload live_analysis_results.csv to the
   'Live Traffic' tab in your dashboard
